In [103]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS, Final_experiments_baseline, Final_experiments_v1
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect, domain_best_by_model_with_baseline_delta, domain_best_by_model, recommended_by_domain_for_model
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature
from titanic_ml.common.data.eda import sample_dataframe


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [104]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["cb08__pclass_sex_features"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     result_df = save_results(exp_result)
#     save_configs(exp_config)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe09__ticket_group_size
Experiment: fe10__fare_per_ticket_member
Experiment: fe11__age_bin
Experiment: fe12__sex_pclass
Experiment: cb01__age_and_bins
Experiment: cb02__age_imputed_title_and_bins
Experiment: cb03__age_imputed_title_Pclass_and_bins
Experiment: cb04__fare_and_fare_per_family
Experiment: cb05__fare_and_fare_per_ticket
Experiment: cb06__all_fare_features
Experiment: cb07__family_features
Experiment: cb08__pclass_sex_features
Experiment: ab01__age_and_bins_without_fare
Experiment: ab02__age_imputed_title_and_bins_without_fare
Experiment: ab03__age_imputed_title_Pclass_and_bins_without_fare


In [105]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [106]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

running exp: {'name': 'cb08__pclass_sex_features__logreg', 'features': ['Pclass', 'Sex', 'SibSp', 'Fare', 'Parch', 'Embarked', 'Age', 'Sex_Pclass'], 'feature_engineering': [<function add_sex_pclass at 0x000001594EE05120>], 'preprocessing': {'numeric_features': ['Fare', 'SibSp', 'Parch', 'Age'], 'onehot_features': ['Sex', 'Embarked', 'Sex_Pclass'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Combo 08: Exploring the effect of using both raw Pclass, sex and their combination.', 'stage': 'cb08', 'feature_group': 'pclass_sex_features', 'group': 'cb08__pclass_sex_features', 'domain': 'pclass_sex'}
running exp: {'name': 'cb08__pclass_sex_features__knn', 'features': ['Pclass', '

In [107]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()

# For combos:
all_results = load_results()
references = ['fe12__sex_pclass']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
### cb08__pclass_sex_features

_Description pending._

<details>
<summary>Conclusion</summary>


#### Interpretation

- Verdict: mixed
- Recommended for specific models:
  - logreg: test_accuracy_mean: 0.016


#### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group             | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:--------------------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | cb08__pclass_sex_features | logreg        |                          0.786 |                        0.802 |                      0.016 

In [108]:
domains = all_results["domain"].unique().tolist()
for domain in domains:
    print(domain_best_by_model_with_baseline_delta(all_results, domain).to_markdown())
    print()

|    | model_name    | experiment                                           | group                                   | domain   |   test_accuracy_mean |   test_f1_mean |   test_accuracy_mean_baseline |   test_accuracy_mean_delta_vs_baseline |
|---:|:--------------|:-----------------------------------------------------|:----------------------------------------|:---------|---------------------:|---------------:|------------------------------:|---------------------------------------:|
|  0 | random_forest | fe11__age_bin__random_forest                         | fe11__age_bin                           | age      |                0.833 |          0.759 |                         0.822 |                                  0.011 |
|  1 | svc           | ab03__age_imputed_title_Pclass_and_bins__svc         | ab03__age_imputed_title_Pclass_and_bins | age      |                0.831 |          0.766 |                         0.827 |                                  0.004 |
|  2 | xgb           | c

In [109]:
print(domain_best_by_model(all_results, 'age'))

      model_name                                         experiment  \
0  random_forest                       fe11__age_bin__random_forest   
1            svc       ab03__age_imputed_title_Pclass_and_bins__svc   
2            xgb              cb02__age_imputed_title_and_bins__xgb   
3    extra_trees  cb03__age_imputed_title_Pclass_and_bins__extra...   
4            knn                                 fe11__age_bin__knn   
5  decision_tree                       fe11__age_bin__decision_tree   
6         logreg    cb03__age_imputed_title_Pclass_and_bins__logreg   

                                     group domain  test_accuracy_mean  \
0                            fe11__age_bin    age               0.833   
1  ab03__age_imputed_title_Pclass_and_bins    age               0.831   
2         cb02__age_imputed_title_and_bins    age               0.828   
3  cb03__age_imputed_title_Pclass_and_bins    age               0.819   
4                            fe11__age_bin    age               0.

In [110]:
def best_by_domain_for_model(
    results_df,
    model_name,
    include_baseline=True,
    only_success=True,
):
    df = results_df.copy()

    if only_success and "status" in df.columns:
        df = df[df["status"] == "success"]

    domain_df = df[df["model_name"] == model_name]

    if include_baseline:
        baseline_df = df[df["group"] == "baseline__raw"]
        domain_df = pd.concat([baseline_df, domain_df], ignore_index=True)

    if domain_df.empty:
        raise ValueError(f"No results found for model: {model_name}")

    best_rows = (
        domain_df
        .sort_values("test_accuracy_mean", ascending=False)
        .groupby("domain", as_index=False)
        .first()
    )

    columns = [
        "model_name",
        "experiment",
        "group",
        "domain",
        "test_accuracy_mean",
        "test_f1_mean",
    ]

    columns = [col for col in columns if col in best_rows.columns]

    return (
        best_rows[columns]
        .sort_values("test_accuracy_mean", ascending=False)
        .reset_index(drop=True)
    )

In [111]:
for model in MODEL_REGISTRY.keys():
    print(f"Best results for model: {model}")
    print(best_by_domain_for_model(all_results, model))
    print()


Best results for model: logreg
  model_name                                         experiment  \
0     logreg                                fe05__title__logreg   
1     logreg    cb03__age_imputed_title_Pclass_and_bins__logreg   
2     logreg                  cb08__pclass_sex_features__logreg   
3     logreg  ab03__age_imputed_title_Pclass_and_bins_withou...   
4     logreg                               fe01__family__logreg   
5     logreg                                 fe03__deck__logreg   
6     logreg               fe08__fare_per_family_member__logreg   
7     logreg                    fe09__ticket_group_size__logreg   

                                               group      domain  \
0                                        fe05__title       title   
1            cb03__age_imputed_title_Pclass_and_bins         age   
2                          cb08__pclass_sex_features  pclass_sex   
3  ab03__age_imputed_title_Pclass_and_bins_withou...    ablation   
4                        

In [112]:
def best_by_domain_for_model_with_baseline(
    results_df,
    model_name,
    metric="test_accuracy_mean", to_markdown=False
):
    best_df = best_by_domain_for_model(
        results_df=results_df,
        model_name=model_name,
        metric=metric,
        include_baseline=True,
    )

    baseline = (
        results_df[results_df["group"] == "baseline__raw"]
        [["model_name", metric]]
        .rename(columns={metric: f"{metric}_baseline"})
    )

    best_df = best_df.merge(baseline, on="model_name", how="left")
    best_df[f"{metric}_delta_vs_baseline"] = (
        best_df[metric] - best_df[f"{metric}_baseline"]
    )
    recommended = list(best_df.loc[best_df[f"{metric}_delta_vs_baseline"]> 0, ["group",f"{metric}_delta_vs_baseline"]].itertuples(index=False, name=None))
    rounded=[]
    for n in range(len(recommended)):
        rounded.append((recommended[n][0],round(recommended[n][1],3)))

    if to_markdown:
        best_df = best_df.to_markdown()

    return {"df":best_df, "recommended":rounded}

In [113]:
def best_by_domain_for_model_with_baseline(
    results_df,
    model_name,
   to_markdown=False
):
    best_df = best_by_domain_for_model(
        results_df=results_df,
        model_name=model_name,
        include_baseline=True,
    )

    baseline = (
        results_df[results_df["group"] == "baseline__raw"]
        [["model_name", "test_accuracy_mean","test_f1_mean"]]
        .rename(columns={"test_accuracy_mean": "test_accuracy_mean_baseline","test_f1_mean":"test_f1_mean_baseline"})
    )

    best_df = best_df.merge(baseline, on="model_name", how="left")
    best_df["test_accuracy_mean_delta_vs_baseline"] = (
        best_df["test_accuracy_mean"] - best_df["test_accuracy_mean_baseline"]
    )
    best_df["test_f1_mean_delta_vs_baseline"] = (
            best_df["test_f1_mean"] - best_df["test_f1_mean_baseline"]
        )
    recommended = list(best_df.loc[best_df["test_accuracy_mean_delta_vs_baseline"]> 0, ["group","test_accuracy_mean_delta_vs_baseline", "test_f1_mean_delta_vs_baseline"]].itertuples(index=False, name=None))
    rounded=[]
    for n in range(len(recommended)):
        rounded.append((recommended[n][0],round(recommended[n][1],3),round(recommended[n][2],3)))

    if to_markdown:
        best_df = best_df.to_markdown()

    return {"df":best_df, "recommended":rounded}

In [114]:
for model in MODEL_REGISTRY.keys():
    best = best_by_domain_for_model_with_baseline(all_results, model, to_markdown=True)
    print(f"Best results for model: {model}")
    print(best['df'])
    print()
    print(best['recommended'])
    print()

Best results for model: logreg
|    | model_name   | experiment                                                   | group                                                | domain     |   test_accuracy_mean |   test_f1_mean |   test_accuracy_mean_baseline |   test_f1_mean_baseline |   test_accuracy_mean_delta_vs_baseline |   test_f1_mean_delta_vs_baseline |
|---:|:-------------|:-------------------------------------------------------------|:-----------------------------------------------------|:-----------|---------------------:|---------------:|------------------------------:|------------------------:|---------------------------------------:|---------------------------------:|
|  0 | logreg       | fe05__title__logreg                                          | fe05__title                                          | title      |                0.825 |          0.765 |                         0.786 |                   0.713 |                                  0.039 |                        

In [115]:
for model in MODEL_REGISTRY.keys():
    best = best_by_domain_for_model_with_baseline(all_results, model, to_markdown=True)
    recommended = best['recommended']
    rec =[ (x,y,z) for x,y,z in recommended if(y > 0.003 and z > -0.01)]
    names = [x for x,y,z in rec]
    print(f"Best results for model: {model}")
    print(best['df'])
    print(rec)
    print(names)
    print()

Best results for model: logreg
|    | model_name   | experiment                                                   | group                                                | domain     |   test_accuracy_mean |   test_f1_mean |   test_accuracy_mean_baseline |   test_f1_mean_baseline |   test_accuracy_mean_delta_vs_baseline |   test_f1_mean_delta_vs_baseline |
|---:|:-------------|:-------------------------------------------------------------|:-----------------------------------------------------|:-----------|---------------------:|---------------:|------------------------------:|------------------------:|---------------------------------------:|---------------------------------:|
|  0 | logreg       | fe05__title__logreg                                          | fe05__title                                          | title      |                0.825 |          0.765 |                         0.786 |                   0.713 |                                  0.039 |                        

In [116]:
thresholds = {
    "accuracy": 0.003,
    "f1": -0.01,
}

for model in MODEL_REGISTRY:

    result = recommended_by_domain_for_model(
        results_df=all_results,
        model_name=model,
        thresholds=thresholds,
    )

    print(f"\nBest candidates for {model}")
    print("=" * 50)

    print("\nRecommended:")
    for recommendation in result["recommended"]:
        print(
            recommendation["domain"],
            "->",
            recommendation["group"],
            recommendation["deltas"],
        )
    print()
    print("recommended list:")
    for recommendation in result["recommended"]:
        print(recommendation["group"], end=", ")
    print()
    print("\nFull domain results:")
    print(result["df"].to_markdown(index=False))


Best candidates for logreg

Recommended:
title -> fe05__title {'accuracy': 0.039, 'f1': 0.052}
age -> cb03__age_imputed_title_Pclass_and_bins {'accuracy': 0.02, 'f1': 0.022}
ablation -> ab03__age_imputed_title_Pclass_and_bins_without_fare {'accuracy': 0.016, 'f1': 0.017}
pclass_sex -> cb08__pclass_sex_features {'accuracy': 0.016, 'f1': 0.0}
family -> fe01__family {'accuracy': 0.009, 'f1': 0.008}
cabin -> fe04__cabin_features {'accuracy': 0.005, 'f1': 0.011}
fare -> fe08__fare_per_family_member {'accuracy': 0.003, 'f1': 0.004}

recommended list:
fe05__title, cb03__age_imputed_title_Pclass_and_bins, ab03__age_imputed_title_Pclass_and_bins_without_fare, cb08__pclass_sex_features, fe01__family, fe04__cabin_features, fe08__fare_per_family_member, 

Full domain results:
| domain     | group                                                | experiment                                                   | model_name   |   test_accuracy_mean |   test_f1_mean |   accuracy_delta_vs_baseline |   f1_

In [117]:
for model in Final_experiments_baseline.keys():
    print(model)

logreg
knn
svc
decision_tree
random_forest
extra_trees
xgb


In [118]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [119]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [120]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [121]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [122]:
# print(workflow["all_results"])

In [123]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

Old Eda practice

In [124]:
# eda = run_eda(train_df, target=TARGET, display=True, head=3,random=4, tail=3)
# print()

In [125]:
# print(eda)

In [126]:
# print(eda['dataFrame_health'].to_markdown())

In [127]:
# print(eda["dataFrame_summary"].to_markdown())

In [128]:
# for col in eda['categorical_summary']:
#     sample = sample_dataframe(eda['categorical_summary'][col], head=1, random=3, tail=1)
#     sample_df = pd.concat(
#         [sample[k] for k in ['head', 'random', 'tail']],
#         ignore_index=False
#     )
#     print(sample_df.to_markdown())
#     print()

In [129]:
# for col in eda['numerical_summary']:
#     print(f"Numerical column: {col}")
#     print(eda['numerical_summary'][col].to_markdown())
#     print()

In [130]:
# print("Correlation matrix:")
# print(eda["correlation_matrix"].to_markdown())

In [131]:
# print('Correlation with the target variable:')
# print(eda["target_correlation"].to_markdown())

In [132]:
# for col in eda["categorical_rare"]:
#     print(f"Categorical column with rare values: {col}")
#     print(eda["categorical_rare"][col])
#     print()

In [133]:
# # print(eda['sample'])
# sample_df = pd.concat(
#     [eda['sample'][k] for k in ['head', 'random', 'tail']],
#     ignore_index=True
# )
# print(sample_df.to_markdown(index=False))